# 01 Data Quality Control & Harmonization Audit (Person 1)

Audits the harmonized **NASA PCoE**, **Oxford Battery Degradation**, and **Sandia (SNL Preger 2020)** bundles built on the canonical 2 mV `[3.500, 4.180] V` grid:
1. Cohort summary table (cells, cycles, nominal capacity, temperature, regeneration/outlier/below-EOL counts).
2. SOH degradation trajectories highlighting flagged **capacity regeneration** jumps and **label outliers**.
3. Incremental Capacity (`dQ/dV`) and Differential Thermal Voltammetry (`dT/dV`) aging evolution on representative cells.
4. Partial-window cycle coverage audit (`[3.60, 4.10] V` through `[3.90, 4.10] V`) illustrating aged-cell ohmic rebound.

> Prerequisites: run `python scripts/download_datasets.py --dataset all` and `python scripts/build_processed.py --dataset all`.

In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = REPO_ROOT / "data" / "processed"

bundles = {}
for key in ["nasa", "oxford", "sandia"]:
    pkl_path = PROCESSED_DIR / f"{key}.pkl"
    if pkl_path.exists():
        with open(pkl_path, "rb") as f:
            bundles[key] = pickle.load(f)
        print(f"Loaded {key}: {len(bundles[key].cell_ids)} cells, {len(bundles[key].cycles)} cycles")
    else:
        print(f"Missing {pkl_path} — run build_processed.py --dataset {key}")


## 1. Cross-Dataset Cohort Summary

In [ ]:
def window_coverage_pct(cycles, v_start, v_end):
    if not cycles:
        return 0.0
    ok = 0
    for c in cycles:
        m = (c.v_grid >= v_start - 1e-6) & (c.v_grid <= v_end + 1e-6)
        if np.any(m) and np.all(np.isfinite(c.q_norm[m])) and np.all(np.isfinite(c.dq_dv[m])):
            ok += 1
    return 100.0 * ok / len(cycles)

rows = []
for key, b in bundles.items():
    sohs = np.array([c.soh_true for c in b.cycles])
    rows.append({
        "dataset": key,
        "n_cells": len(b.cell_ids),
        "n_cycles": len(b.cycles),
        "cycle_unit": b.metadata.get("cycle_index_unit", "cycles"),
        "soh_min_pct": round(float(sohs.min()), 2),
        "soh_max_pct": round(float(sohs.max()), 2),
        "n_regen": sum(c.is_regeneration for c in b.cycles),
        "n_outliers": sum(c.is_label_outlier for c in b.cycles),
        "below_eol_pct": round(100.0 * sum(c.is_below_eol for c in b.cycles) / len(b.cycles), 1),
        "cov_3.60_4.10_pct": round(window_coverage_pct(b.cycles, 3.60, 4.10), 1),
        "cov_3.85_4.10_pct": round(window_coverage_pct(b.cycles, 3.85, 4.10), 1),
        "cov_3.90_4.10_pct": round(window_coverage_pct(b.cycles, 3.90, 4.10), 1),
    })

summary_df = pd.DataFrame(rows)
summary_df


## 2. SOH Trajectories (Regeneration Jumps & Label Outliers Flagged, Not Dropped)

In [ ]:
fig, axes = plt.subplots(1, len(bundles), figsize=(6 * len(bundles), 4.5), sharey=True)
if len(bundles) == 1:
    axes = [axes]

for ax, (key, b) in zip(axes, bundles.items()):
    plot_cells = b.cell_ids if len(b.cell_ids) <= 8 else [
        "SNL_18650_NCA_25C_0-100_0.5-1C_a",
        "SNL_18650_NCA_15C_0-100_0.5-2C_a",
        "SNL_18650_NMC_25C_0-100_0.5-1C_a",
        "SNL_18650_NMC_15C_0-100_0.5-2C_b",
    ]
    for cid in plot_cells:
        cc = [c for c in b.cycles if c.cell_id == cid]
        if not cc:
            continue
        x = [c.cycle_index for c in cc]
        y = [c.soh_true for c in cc]
        ax.plot(x, y, lw=1.4, label=cid)
        regen = [(c.cycle_index, c.soh_true) for c in cc if c.is_regeneration]
        if regen:
            ax.scatter(*zip(*regen), marker="^", s=24, color="green", zorder=5)
        out = [(c.cycle_index, c.soh_true) for c in cc if c.is_label_outlier]
        if out:
            ax.scatter(*zip(*out), marker="x", s=36, color="red", zorder=6)
    ax.axhline(80.0, ls="--", color="gray", lw=1.0, label="80% EOL")
    ax.set_title(f"{key.upper()} ({b.metadata.get('cycle_index_unit', 'cycles')})")
    ax.set_xlabel("Cycle index")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7, loc="lower left")

axes[0].set_ylabel("SOH (%)")
fig.suptitle("SOH Degradation Curves (green ^ = regeneration, red x = label outlier)")
plt.tight_layout()
plt.show()


## 3. IC (`dQ/dV`) & DTV (`dT/dV`) Curve Evolution Across Aging

In [ ]:
rep_cells = [
    ("nasa", "B0005"),
    ("oxford", "Cell1"),
    ("sandia", "SNL_18650_NCA_25C_0-100_0.5-1C_a"),
    ("sandia", "SNL_18650_NMC_25C_0-100_0.5-1C_a"),
]
rep_cells = [(k, cid) for k, cid in rep_cells if k in bundles]

fig, axes = plt.subplots(2, len(rep_cells), figsize=(4.5 * len(rep_cells), 7), sharex=True)
cmap = plt.cm.viridis

for col, (key, cid) in enumerate(rep_cells):
    cc = [c for c in bundles[key].cycles if c.cell_id == cid]
    idxs = np.linspace(1, len(cc) - 1, 6, dtype=int)
    for j, i in enumerate(idxs):
        c = cc[i]
        color = cmap(j / max(len(idxs) - 1, 1))
        lbl = f"cyc {c.cycle_index} ({c.soh_true:.1f}%)"
        axes[0, col].plot(c.v_grid, c.dq_dv, color=color, lw=1.3, label=lbl)
        axes[1, col].plot(c.v_grid, c.dt_dv, color=color, lw=1.3)
    axes[0, col].set_title(f"{key.upper()}: {cid}", fontsize=9)
    axes[0, col].legend(fontsize=7)
    axes[0, col].grid(alpha=0.3)
    axes[1, col].grid(alpha=0.3)
    axes[1, col].set_xlabel("Voltage (V)")

axes[0, 0].set_ylabel("dQ_norm / dV (1/V)")
axes[1, 0].set_ylabel("dT / dV (°C/V)")
plt.tight_layout()
plt.show()


## 4. Partial-Window Coverage Audit (Zero-Extrapolation Integrity)

In [ ]:
windows = [(3.60, 4.10), (3.75, 4.10), (3.85, 4.10), (3.90, 4.10)]
win_labels = [f"[{s:.2f}, {e:.2f}] V" for s, e in windows]

cov_records = []
for key, b in bundles.items():
    for cid in b.cell_ids:
        cc = [c for c in b.cycles if c.cell_id == cid]
        rec = {"dataset": key, "cell_id": cid}
        for (s, e), lbl in zip(windows, win_labels):
            rec[lbl] = window_coverage_pct(cc, s, e)
        cov_records.append(rec)

cov_df = pd.DataFrame(cov_records)
min_cov = cov_df.groupby("dataset")[win_labels].min().round(1)
print("Minimum per-cell cycle coverage (%) per dataset:")
min_cov
